In [10]:
import pandas as pd
import numpy as np

# Load complete invoice history
invoices = pd.read_csv(
    "C:\\Users\\navee\\Downloads\\churnguard-ai\\data\\processed\\invoice_level.csv",
    parse_dates=["invoice_date"]
)

print("Dataset shape:", invoices.shape)
print("\nColumns:")
print(invoices.columns.tolist())

print("\nDate range:")
print(invoices["invoice_date"].min())
print(invoices["invoice_date"].max())

print("\nUnique customers:", invoices["customer_id"].nunique())
print("Unique invoices:", invoices["invoice"].nunique())

Dataset shape: (36969, 6)

Columns:
['invoice', 'customer_id', 'invoice_date', 'invoice_value', 'total_quantity', 'distinct_products']

Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00

Unique customers: 5878
Unique invoices: 36969


In [11]:
# Historical prediction cutoff dates
cutoffs = pd.to_datetime([
    "2010-06-01",
    "2010-09-01",
    "2010-12-01",
    "2011-03-01",
    "2011-06-01",
    "2011-09-01"
])

snapshots = []

for cutoff in cutoffs:

    feature_start = cutoff - pd.Timedelta(days=90)
    label_end = cutoff + pd.Timedelta(days=90)

    # Purchases in the 90 days before the cutoff
    feature_data = invoices[
        (invoices["invoice_date"] >= feature_start) &
        (invoices["invoice_date"] < cutoff)
    ].copy()

    # Purchases in the 90 days after the cutoff
    future_data = invoices[
        (invoices["invoice_date"] >= cutoff) &
        (invoices["invoice_date"] < label_end)
    ]

    # Build features using only historical information
    features_df = feature_data.groupby("customer_id").agg(
        orders_90d=("invoice", "nunique"),
        spend_90d=("invoice_value", "sum"),
        quantity_90d=("total_quantity", "sum"),
        last_purchase=("invoice_date", "max")
    ).reset_index()

    features_df["recency_days"] = (
        cutoff - features_df["last_purchase"]
    ).dt.days

    # Customers with a purchase in the future window are retained
    future_customers = set(future_data["customer_id"].unique())

    features_df["churn_label"] = (
        ~features_df["customer_id"].isin(future_customers)
    ).astype(int)

    features_df["cutoff_date"] = cutoff

    features_df = features_df.drop(columns=["last_purchase"])

    snapshots.append(features_df)

# Combine the snapshots
snapshot_df = pd.concat(snapshots, ignore_index=True)

print("Snapshot dataset shape:", snapshot_df.shape)

print("\nCustomers per cutoff:")
print(snapshot_df.groupby("cutoff_date").size())

print("\nChurn rate per cutoff:")
print(
    snapshot_df.groupby("cutoff_date")["churn_label"]
    .mean()
    .round(3)
)

display(snapshot_df.head())

Snapshot dataset shape: (12400, 7)

Customers per cutoff:
cutoff_date
2010-06-01    2023
2010-09-01    1936
2010-12-01    2877
2011-03-01    1682
2011-06-01    1959
2011-09-01    1923
dtype: int64

Churn rate per cutoff:
cutoff_date
2010-06-01    0.442
2010-09-01    0.313
2010-12-01    0.579
2011-03-01    0.433
2011-06-01    0.435
2011-09-01    0.319
Name: churn_label, dtype: float64


,customer_id,orders_90d,spend_90d,quantity_90d,recency_days,churn_label,cutoff_date
0,12349,2,1268.52,474,13,1,2010-06-01
1,12355,1,488.21,303,10,1,2010-06-01
2,12359,2,683.34,168,69,0,2010-06-01
3,12360,2,662.04,656,6,1,2010-06-01
4,12361,1,98.15,38,22,1,2010-06-01


In [12]:

# Chronological validation and final test setup
import pandas as pd
import numpy as np

validation_cutoff = pd.Timestamp("2011-06-01")
test_cutoff = pd.Timestamp("2011-09-01")

feature_columns = [
    "orders_90d",
    "spend_90d",
    "quantity_90d",
    "recency_days"
]

# Validation training data: labels must be fully observed
# before the validation cutoff.
validation_train = snapshot_df[
    (snapshot_df["cutoff_date"] < validation_cutoff) &
    (
        snapshot_df["cutoff_date"] + pd.Timedelta(days=90)
        <= validation_cutoff
    )
].copy()

# Validation snapshot: used to select the threshold.
validation_snapshot = snapshot_df[
    snapshot_df["cutoff_date"] == validation_cutoff
].copy()

# Final training data: labels must be fully observed
# before the final test cutoff.
train_snapshots = snapshot_df[
    (snapshot_df["cutoff_date"] < test_cutoff) &
    (
        snapshot_df["cutoff_date"] + pd.Timedelta(days=90)
        <= test_cutoff
    )
].copy()

# Final test snapshot: keep separate for final evaluation.
test_snapshot = snapshot_df[
    snapshot_df["cutoff_date"] == test_cutoff
].copy()

X_val_train = validation_train[feature_columns]
y_val_train = validation_train["churn_label"]

X_validation = validation_snapshot[feature_columns]
y_validation = validation_snapshot["churn_label"]

X_train_time = train_snapshots[feature_columns]
y_train_time = train_snapshots["churn_label"]

X_test_time = test_snapshot[feature_columns]
y_test_time = test_snapshot["churn_label"]

print("Validation training rows:", len(X_val_train))
print("Validation rows:", len(X_validation))
print("Final training rows:", len(X_train_time))
print("Final test rows:", len(X_test_time))

print("\nValidation training cutoffs:")
print(sorted(validation_train["cutoff_date"].dt.strftime("%Y-%m-%d").unique()))

print("\nFinal training cutoffs:")
print(sorted(train_snapshots["cutoff_date"].dt.strftime("%Y-%m-%d").unique()))

print("\nValidation churn rate:", round(y_validation.mean(), 3))
print("Final test churn rate:", round(y_test_time.mean(), 3))


Validation training rows: 8518
Validation rows: 1959
Final training rows: 10477
Final test rows: 1923

Validation training cutoffs:
['2010-06-01', '2010-09-01', '2010-12-01', '2011-03-01']

Final training cutoffs:
['2010-06-01', '2010-09-01', '2010-12-01', '2011-03-01', '2011-06-01']

Validation churn rate: 0.435
Final test churn rate: 0.319


In [13]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

# 1. Train a temporary model using earlier historical data.
validation_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

validation_model.fit(X_val_train, y_val_train)

# 2. Select threshold using validation predictions only.
validation_probabilities = validation_model.predict_proba(
    X_validation
)[:, 1]

threshold_results = []

for threshold in np.arange(0.20, 0.71, 0.05):
    predictions = (validation_probabilities >= threshold).astype(int)

    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "precision": precision_score(
            y_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_validation, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_validation, predictions, zero_division=0
        )
    })

validation_thresholds = pd.DataFrame(threshold_results)
display(validation_thresholds.round(3))

best_row = validation_thresholds.loc[
    validation_thresholds["f1"].idxmax()
]
best_threshold = float(best_row["threshold"])

print("Selected validation threshold:", best_threshold)
print("Validation F1:", round(float(best_row["f1"]), 3))

# 3. Train the final model on all eligible earlier snapshots.
rf_time = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_time.fit(X_train_time, y_train_time)

# 4. Evaluate once on the final September test snapshot.
rf_time_probabilities = rf_time.predict_proba(X_test_time)[:, 1]
rf_time_predictions = (
    rf_time_probabilities >= best_threshold
).astype(int)

dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train_time, y_train_time)
dummy_predictions = dummy_model.predict(X_test_time)

print("\nDUMMY BASELINE")
print("Accuracy:", round(accuracy_score(y_test_time, dummy_predictions), 3))

print("\nFINAL CHRONOLOGICAL RANDOM FOREST")
print("Threshold:", best_threshold)
print("Accuracy:", round(accuracy_score(y_test_time, rf_time_predictions), 3))
print("Precision:", round(precision_score(y_test_time, rf_time_predictions, zero_division=0), 3))
print("Recall:", round(recall_score(y_test_time, rf_time_predictions, zero_division=0), 3))
print("F1-score:", round(f1_score(y_test_time, rf_time_predictions, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_test_time, rf_time_probabilities), 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test_time, rf_time_predictions))

print("\nClassification report:")
print(classification_report(
    y_test_time, rf_time_predictions, zero_division=0
))


,threshold,precision,recall,f1
0,0.20,0.473,0.989,0.640
1,0.25,0.489,0.975,0.652
2,0.30,0.500,0.958,0.657
3,0.35,0.512,0.940,0.663
4,0.40,0.534,0.902,0.671
5,0.45,0.550,0.862,0.671
6,0.50,0.580,0.816,0.678
7,0.55,0.591,0.749,0.661
8,0.60,0.624,0.580,0.601
9,0.65,0.668,0.317,0.430


Selected validation threshold: 0.5
Validation F1: 0.678

DUMMY BASELINE
Accuracy: 0.681

FINAL CHRONOLOGICAL RANDOM FOREST
Threshold: 0.5
Accuracy: 0.591
Precision: 0.424
Recall: 0.791
F1-score: 0.552
ROC-AUC: 0.704

Confusion matrix:
[[652 658]
 [128 485]]

Classification report:
              precision    recall  f1-score   support

           0       0.84      0.50      0.62      1310
           1       0.42      0.79      0.55       613

    accuracy                           0.59      1923
   macro avg       0.63      0.64      0.59      1923
weighted avg       0.70      0.59      0.60      1923



In [14]:

# Final test results at the threshold selected on validation data.
# Do not use these test results to choose a new threshold.

print("Validation-selected threshold:", best_threshold)
print("Final test customers:", len(y_test_time))
print("Final test churn rate:", round(y_test_time.mean(), 3))
print("Final test predicted churners:", int(rf_time_predictions.sum()))


Validation-selected threshold: 0.5
Final test customers: 1923
Final test churn rate: 0.319
Final test predicted churners: 1143


In [15]:
import pandas as pd
import numpy as np

# Customer-level predictions from chronological Random Forest
roi_df = test_snapshot[
    ["customer_id", "churn_label"]
].copy()

roi_df["churn_probability"] = rf_time_probabilities

print("Customers available:", len(roi_df))
print(roi_df.head())

Customers available: 1923
       customer_id  churn_label  churn_probability
10477        12347            0           0.331615
10478        12358            1           0.592707
10479        12359            0           0.579756
10480        12360            0           0.603695
10481        12362            0           0.355692


In [16]:
def retention_roi(
    customers,
    offer_cost,
    success_rate,
    value_per_saved_customer,
    threshold
):
    # Select customers at or above the risk threshold
    selected = customers[
        customers["churn_probability"] >= threshold
    ]

    contacted = len(selected)

    campaign_cost = contacted * offer_cost

    # Scenario assumption: the offer prevents churn for this share
    # of customers who would otherwise churn.
    expected_saves = (
        selected["churn_label"].sum() * success_rate
    )

    expected_value_saved = (
        expected_saves * value_per_saved_customer
    )

    net_value = expected_value_saved - campaign_cost

    return {
        "threshold": threshold,
        "customers_contacted": contacted,
        "campaign_cost": round(campaign_cost, 2),
        "expected_saves": round(expected_saves, 2),
        "expected_value_saved": round(expected_value_saved, 2),
        "estimated_net_value": round(net_value, 2)
    }


# Example business assumptions — change these later
result = retention_roi(
    customers=roi_df,
    offer_cost=100,
    success_rate=0.20,
    value_per_saved_customer=2000,
    threshold=0.40
)

display(pd.DataFrame([result]))

,threshold,customers_contacted,campaign_cost,expected_saves,expected_value_saved,estimated_net_value
0,0.4,1339,133900,109.0,218000.0,84100.0


In [17]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

roi_results = []

for threshold in thresholds:
    roi_results.append(
        retention_roi(
            customers=roi_df,
            offer_cost=100,
            success_rate=0.20,
            value_per_saved_customer=2000,
            threshold=threshold
        )
    )

roi_comparison = pd.DataFrame(roi_results)

display(roi_comparison)

,threshold,customers_contacted,campaign_cost,expected_saves,expected_value_saved,estimated_net_value
0,0.3,1569,156900,117.2,234400.0,77500.0
1,0.4,1339,133900,109.0,218000.0,84100.0
2,0.5,1143,114300,97.0,194000.0,79700.0
3,0.6,680,68000,64.2,128400.0,60400.0
4,0.7,201,20100,23.4,46800.0,26700.0


In [18]:
%pip install joblib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\navee\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [19]:
import os
import joblib

# Create a folder for saved models
os.makedirs("models", exist_ok=True)

# Save the trained chronological Random Forest
model_package = {
    "model": rf_time,
    "features": feature_columns
}

joblib.dump(
    model_package,
    "models/churn_random_forest.joblib"
)

print("Model saved successfully!")
print("Path: models/churn_random_forest.joblib")

Model saved successfully!
Path: models/churn_random_forest.joblib


In [21]:

from pathlib import Path
import joblib

BASE_DIR = Path.cwd()
MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "churn_random_forest.joblib"

model_package = {
    "model": rf_time,
    "features": feature_columns,
    "threshold": 0.50
}

joblib.dump(model_package, model_path)

print("Model saved successfully!")
print("Path:", model_path.resolve())
print("File exists:", model_path.exists())


Model saved successfully!
Path: C:\Users\navee\Downloads\churnguard-ai\notebooks\models\churn_random_forest.joblib
File exists: True


In [22]:

saved_model = joblib.load(model_path)

print("Model:", type(saved_model["model"]).__name__)
print("Features:", saved_model["features"])
print("Threshold:", saved_model["threshold"])


Model: RandomForestClassifier
Features: ['orders_90d', 'spend_90d', 'quantity_90d', 'recency_days']
Threshold: 0.5
